<a href="https://colab.research.google.com/github/jaggu1367/data-quality-inspector/blob/demo/GE_demo1_draft.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [34]:
import sys
sys.version

'3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]'

In [35]:
from pyspark.sql import SparkSession
import great_expectations as gx

In [36]:
print(gx.__version__)

1.11.3


In [2]:

!pip install great-expectations pyspark

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 813.6/813.6 kB 41.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 10.3 MB/s eta 0:00:00
  Attempting uninstall: altair
    Found existing installation: altair 5.5.0
    Uninstalling altair-5.5.0:
      Successfully uninstalled altair-5.5.0


In [22]:
context = gx.get_context(mode="file")

# Optional. Request a File Data Context from a specific folder.
context = gx.get_context(mode="file", project_root_dir="./ge")

In [24]:
# Define the Data Source name
data_source_name = "temp_data_source_name"
# Add the Data Source to the Data Context
data_source = context.data_sources.add_spark(data_source_name)

In [25]:
# Retrieve the Data Source
data_source_name = "temp_data_source_name"
data_source = context.data_sources.get(data_source_name)

# Define the Data Asset name
data_asset_name = "temp_dataframe_data_asset"

# Add a Data Asset to the Data Source
data_asset = data_source.add_dataframe_asset(name=data_asset_name)

In [27]:
# Retrieve the Data Asset
data_source_name = "temp_data_source_name"
data_asset_name = "temp_dataframe_data_asset"
data_asset = context.data_sources.get(data_source_name).get_asset(data_asset_name)

# Define the Batch Definition name
batch_definition_name = "temp_batch_definition"

# Add a Batch Definition to the Data Asset
batch_definition = data_asset.add_batch_definition_whole_dataframe(
    batch_definition_name
)

In [28]:
# Provide a dataframe through Batch Parameters
spark = SparkSession.builder.getOrCreate()
df = spark.createDataFrame([
    {"bonus": 6000, "tenure": 3, "salary": 45000, "department": "Sales", "commission": 1500},
    {"bonus": 8000, "tenure": 1, "salary": 60000, "department": "HR", "commission": 1800},
    {"bonus": 9500, "tenure": 4, "salary": 40000, "department": "Engineering", "commission": 1200},
])
batch_parameters = {"dataframe": df}

In [31]:
# Create an Expectation to test
expectation = gx.expectations.ExpectColumnValuesToBeBetween(
    column="bonus", max_value=10000, min_value=6000
)

In [32]:

# Get the dataframe as a Batch
batch = batch_definition.get_batch(batch_parameters=batch_parameters)

# Test the Expectation
validation_results = batch.validate(expectation)
print(validation_results)

Calculating Metrics:   0%|          | 0/13 [00:00<?, ?it/s]

{
  "success": true,
  "expectation_config": {
    "type": "expect_column_values_to_be_between",
    "kwargs": {
      "batch_id": "temp_data_source_name-temp_dataframe_data_asset",
      "column": "bonus",
      "min_value": 6000.0,
      "max_value": 10000.0
    },
    "meta": {},
    "severity": "critical"
  },
  "result": {
    "element_count": 3,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": [],
    "missing_count": 0,
    "missing_percent": 0.0,
    "unexpected_percent_total": 0.0,
    "unexpected_percent_nonmissing": 0.0,
    "partial_unexpected_counts": []
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}


In [26]:
SQL_EXPECTATIONS = [
    "bonus BETWEEN 5000 AND 10000 AND (tenure > 2 AND salary <= 50000) OR (department = 'Sales')",
    "commission BETWEEN 1000 AND 2000",
    "department IN ('Sales','HR','Engineering')",
    "salary > 30000"
]

In [18]:
import sqlglot
import great_expectations as gx

def translate_sql_to_expectations(sql_expr: str, ge_df):
    parsed = sqlglot.parse_one(f"SELECT {sql_expr}")
    expectations = []

    def handle_expr(expr):
        if expr.token_type == "BETWEEN":
            col = expr.this.name
            low = int(expr.args["low"].name)
            high = int(expr.args["high"].name)
            expectations.append(
                ge_df.expect_column_values_to_be_between(col, min_value=low, max_value=high)
            )
        elif expr.token_type == "EQ":
            col = expr.left.name
            val = expr.right.name.strip("'")
            expectations.append(
                ge_df.expect_column_values_to_equal(col, val)
            )
        elif expr.token_type == "IN":
            col = expr.this.name
            vals = [v.name.strip("'") for v in expr.expressions]
            expectations.append(
                ge_df.expect_column_values_to_be_in_set(col, vals)
            )
        elif expr.token_type in ("GT", "LT", "GTE", "LTE"):
            col = expr.left.name
            val = int(expr.right.name)
            if expr.token_type == "GT":
                expectations.append(ge_df.expect_column_values_to_be_between(col, min_value=val+1))
            elif expr.token_type == "GTE":
                expectations.append(ge_df.expect_column_values_to_be_between(col, min_value=val))
            elif expr.token_type == "LT":
                expectations.append(ge_df.expect_column_values_to_be_between(col, max_value=val-1))
            elif expr.token_type == "LTE":
                expectations.append(ge_df.expect_column_values_to_be_between(col, max_value=val))
        # Nested AND/OR handled by recursion
        if hasattr(expr, "left") and hasattr(expr, "right"):
            handle_expr(expr.left)
            handle_expr(expr.right)

    handle_expr(parsed.expressions[0])
    return expectations